In [37]:
import numpy as np
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [11]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
df = pd.read_csv(
    '/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv',
    encoding='latin-1',
    header=None
)
df = df.sample(50000)

In [13]:
df.columns = [
    'target',
    'id',
    'date',
    'flag',
    'user',
    'text'
]

In [14]:
df = df[['target', 'text']]

In [15]:
df['target'] = df['target'].replace(4, 1)

In [16]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    text = text.lower()
    text = text.split()
    
    text = [
        stemmer.stem(word)
        for word in text
        if word not in stop_words
    ]
    
    return " ".join(text)

In [17]:
df['clean_text'] = df['text'].apply(clean_text)
df['clean_text']

1454112    wow mobilis sure know ralli peopl caus lt foll...
1090848            thank school graduat dri mango parti care
374984                                                      
492248     aw thank im bout head bed next min enjoy beer ...
10911                      lmao know stripper want play miss
                                 ...                        
398905                            know everyon birthday mine
1132325                                            haha know
1378201    go snuggl iphon hubbi noel tommorrow beau th b...
529179                 oh knew gone ms own hell ie critic go
1290825                                            good morn
Name: clean_text, Length: 50000, dtype: object

In [18]:
X = df['clean_text']

y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)



In [19]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train = vectorizer.fit_transform(X_train)

X_test = vectorizer.transform(X_test)



In [20]:
model = LogisticRegression()

model.fit(X_train, y_train)



LogisticRegression()

In [21]:
y_pred = model.predict(X_test)


In [22]:
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.7503


In [30]:
tweet = input("\nEnter a Tweet: ")




Enter a Tweet:  I love cat but it bites me


In [31]:
cleaned_tweet = clean_text(tweet)



In [32]:
tweet_vector = vectorizer.transform([cleaned_tweet])



In [33]:
prediction = model.predict(tweet_vector)


In [34]:
polarity = TextBlob(cleaned_tweet).sentiment.polarity


In [35]:
if prediction[0] == 1:
    sentiment = "Positive"
else:
    sentiment = "Negative"

In [36]:
print("\n===== FINAL RESULT =====")

print("Original Tweet:", tweet)

print("Cleaned Tweet:", cleaned_tweet)

print("Polarity Score:", polarity)

print("Predicted Sentiment:", sentiment)


===== FINAL RESULT =====
Original Tweet: I love cat but it bites me
Cleaned Tweet: love cat bite
Polarity Score: 0.5
Predicted Sentiment: Positive
